<a href="https://colab.research.google.com/github/matheus-pvds/TIA_FILES/blob/main/Projeto_Integrador.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import requests
import bs4
import pandas as pd


def categorias():
  dados_municipio = {
    "Organizacional": {
        "Unidades": "Unidades que constituem a estrutura organizacional do município e suas respectivas informações.",
        "Diários Eletrônicos": "Edições do diário eletrônico com todas as publicações oficiais do município.",
        "Guias de Serviços": "Serviços prestados pelo órgão, com todas as especificações sobre a solicitação de cada um deles."
    },
    "Contratos , Parcerias e Licitações": {
        "Contratos": "Contratos firmados pelo órgão, provenientes de compras e prestação de serviços.",
        "Parcerias OSC": "Parcerias celebradas com Organizações da Sociedade Civil (OSC), com respectivas prestações de contas.",
        "Licitações": "Processos licitatórios, com editais, resultados, contratos e outras informações relacionadas."
    },
    "Orçamentária": {
        "Despesas": "Valores empenhados, liquidados e pagos, além de outras informações referentes às despesas.",
        "Receitas": "Valores previstos e realizados, além de outras informações referentes às receitas.",
        "Receitas Mês": "Informações sobre os valores de receita a cada mês.",
        "Receitas Dia": "Informações sobre os valores de receita a cada dia.",
        "Empenhos": "Empenhos realizados pelo órgão e suas respectivas informações.",
        "Emenda Parlamentar": "Emendas parlamentares e suas respectivas informações.",
        "Liquidações": "Liquidações realizadas pelo órgão e suas respectivas informações.",
        "Pagamentos": "Pagamentos realizados pelo órgão e suas respectivas informações.",
        "Extra-orçamentárias": "Receitas e despesas extra-orçamentárias e suas respectivas informações.",
        "Restos a pagar": "Dados referentes às informações de restos a pagar do órgão."
    },
    "Covid-19": {
        "Boletim Epidemiológicos": "Informações sobre o boletim epidemiológico.",
        "Receitas Dia": "Informações sobre os valores de receita a cada dia.",
        "Despesas": "Valores empenhados, liquidados e pagos, além de outras informações referentes às despesas.",
        "Empenhos": "Empenhos realizados pelo órgão e suas respectivas informações.",
        "Liquidações": "Liquidações realizadas pelo órgão e suas respectivas informações.",
        "Pagamentos": "Pagamentos realizados pelo órgão e suas respectivas informações."
    },
    "Recursos Humanos": {
        "Servidores": "Pagamentos, descontos, proventos e demais informações a respeito dos gastos com pessoal."
    },
    "Legislação e Contas Públicas": {
        "Legislações": "Leis, decretos, portarias, e todas as outras legislações municipais.",
        "Contas Públicas": "Relatórios de contas públicas, com seus respectivos arquivos e informações relacionadas."
    },
    "Obras": {
        "Obras": "Obras e demais ações e empreendimentos do município, em andamento e concluídos."
    },
    "Processos Legislativos (Câmaras Municipais)": {
        "Parlamentares": "Parlamentares que compõem cada legislatura da Câmara Municipal, com as informações relacionadas a eles.",
        "Mesa Diretora": "Mesa diretora de cada legislatura da Câmara Municipal.",
        "Comissão": "Comissões de cada legislatura da Câmara Municipal.",
        "Matéria Legislativa": "Matérias legislativas da Câmara Municipal, com seus respectivos trâmites e dados."
    },
    "Processos Seletivos": {
        "Processos Seletivos": "Processos seletivos e concursos realizados pelo órgão com todas as informações relacionadas.",
        "Resultado": "Resultados dos processos seletivos realizados pelo órgão.",
        "Convocação": "Convocação dos candidatos que foram selecionados por processos seletivo"
    }}
  return dados_municipio

def loop_lista(key:str=None):
  dados_municipio:dict=categorias()
  cat = []
  if key:
    for key2 in dados_municipio[key]:
      cat.append(key2)
  else:
    for key in dados_municipio:
      cat.append(key)
  return cat

def resposta():
  cat = loop_lista()
  answer=''
  while answer not in cat:
    print('Utilize uma das categorias:\n')
    for key in cat: print(key)
    answer = input('qual categoria você deseja? ')
  cat = loop_lista(answer)
  while answer not in cat:
    print('Utilize uma das categorias:\n')
    for key in cat: print(key)
    answer = input('qual categoria você deseja? ')
  return answer


def busca_dados(mesBarraAno:str, categoria:str=None, pagina:str='1', idCidade:str='94', tipo:str='json', tamanhoPagina:str='100'):
  if not categoria:
    categoria = resposta()
  endpoint = f'{categoria}'
  url = f'https://dadosabertos-portalfacil.azurewebsites.net/api/{endpoint}'
  params = {
      'idCliente': idCidade,
      'type': tipo,
      'Page': pagina,
      'PageSize': tamanhoPagina,
      'numAno': mesBarraAno
  }

  response = requests.get(url, params=params)

  try:
    return response.json()
  except:
    return response.text

def make_df(mesBarraAno:str, categoria:str=None):
  i = 0
  var = True
  pages = []

  while var:
    i+=1
    print(f'{i}... ', end='')
    page_dict = busca_dados(mesBarraAno=mesBarraAno, categoria=categoria, pagina=f'{i}')

    if page_dict == 'Dados não encontrados':
      if i == 1:
        return 'Dados não encontrados'
      var = False
      return df

    else:

      if i == 1:
        df = pd.DataFrame(page_dict)

      else:
        df2 = pd.DataFrame(page_dict)
        df = pd.concat([df, df2], ignore_index=True)


In [3]:
import re
import pandas as pd
from typing import List, Dict, Any

def transmutar_lista_cargos(lista_cargos: List[str]) -> List[Dict[str, Any]]:
    registros_processados = []

    for item in lista_cargos:
        texto_original = str(item).strip()
        texto_upper = texto_original.upper()

        # 1. Extrair a Carga Horária / Escala (40H, 12X36, ou números isolados no fim)
        carga_match = re.search(r'(\d+X\d+|\d+\s*H|\b\d+\b)', texto_upper)
        carga_horaria = carga_match.group(1) if carga_match else "NÃO ESPECIFICADO"

        # 2. Aplicar os cortes sequenciais no nome do cargo
        # Runa A: Corta tudo à frente do parênteses '('
        nome_limpo = re.sub(r'\(.*', '', texto_upper)

        # Runa B: Remove contagens residuais de strings (ex: ': 1')
        nome_limpo = re.sub(r':\s*\d+', '', nome_limpo)

        # Runa C: Remove Ranks em Algarismos Romanos Isolados (I a V)
        nome_limpo = re.sub(r'\b(I|II|III|IV|V)\b', '', nome_limpo)

        # 🌟 RUNA C.2 (NOVA): Expurgar identificadores de Turma/Ano (Ex: T2016, T201)
        # \bT  -> Procura a letra T isolada no início do token
        # \d{3,4} -> Seguda por exatamente 3 ou 4 dígitos numéricos (anos ou códigos de turma)
        # \b   -> Delimita o fim do token para não quebrar palavras legítimas que comecem com T
        nome_limpo = re.sub(r'\bT\d{3,4}\b', '', nome_limpo)

        # Runa D: Remove as horas e números do corpo do texto principal
        nome_limpo = re.sub(r'\d+X\d+|\d+\s*H|\b\d+\b', '', nome_limpo)

        # Runa E: Substitui hífens por espaços e colapsa espaços duplicados nas pontas
        nome_limpo = nome_limpo.replace('-', ' ')
        nome_limpo = re.sub(r'\s+', ' ', nome_limpo).strip()

        # 3. Empacota o resultado no contrato estável
        registros_processados.append({
            "cargo_original": texto_original,
            "txt_cargo_base": nome_limpo,
            "val_carga_horaria": carga_horaria
        })

    return registros_processados

In [4]:
df = make_df('04/2026', 'Servidores')

df.head()

1... 2... 3... 4... 5... 6... 7... 8... 9... 10... 11... 12... 13... 14... 15... 16... 17... 18... 19... 20... 21... 22... 23... 24... 25... 26... 27... 28... 29... 30... 31... 32... 33... 34... 35... 36... 37... 38... 39... 40... 41... 42... 43... 44... 45... 46... 47... 48... 49... 50... 51... 52... 53... 54... 55... 56... 57... 58... 59... 60... 61... 62... 63... 64... 65... 66... 67... 68... 69... 70... 71... 72... 73... 74... 75... 76... 77... 78... 79... 80... 81... 82... 83... 84... 85... 86... 87... 88... 89... 90... 91... 92... 93... 94... 95... 96... 97... 98... 99... 100... 101... 

,numMatricula,descServidor,descUnidade,descCargo,descFuncao,descVinculo,descTpFolha,descSituacaoContrato,covid,dtCompetencia,vlBase,vlProvento,vlDesconto,vlLiquido,custom,itens
0,81334006,ABDIAS ALVES DOS SANTOS,SECRETARIA MUNICIPAL EDUCACAO,VIGIA DE ESCOLA 12X36,,CONTRATO,FOLHA DO MÊS,ATIVO,False,04/2026,0,2212.32,174.78,2037.54,[],[]
1,55696301,ABEL CAMARA GOMES,SECRETARIA M. DE PLANEJAMENTO E COORDENACAO,ASSISTENTE TECNICO DE SECRETARIA 40H,,EFETIVO/COMISSIONADO,FOLHA DO MÊS,ATIVO,False,04/2026,0,13604.87,4817.99,8786.88,[],[]
2,56687001,ABEL LANA NUNES,SECRETARIA MUNICIPAL DE OBRAS E SERVICOS URBANOS,FISCAL DE POSTURA 40H,,EFETIVO,FOLHA DO MÊS,ATIVO,False,04/2026,0,20448.37,5870.46,14577.91,[],[]
3,12998401,ABELARDO PEREIRA DA SILVA NETO,SECRETARIA MUNICIPAL SAUDE,TECNICO EM RADIOLOGIA 24H,,EFETIVO,FOLHA DO MÊS,ATIVO,False,04/2026,0,9239.67,3758.61,5481.06,[],[]
4,12998402,ABELARDO PEREIRA DA SILVA NETO,SECRETARIA MUNICIPAL SAUDE,ENFERMEIRO 12X36,,EFETIVO,FOLHA DO MÊS,ATIVO,False,04/2026,0,6275.70,1744.54,4531.16,[],[]


In [5]:
for column in df.columns:
  print(column)

numMatricula
descServidor
descUnidade
descCargo
descFuncao
descVinculo
descTpFolha
descSituacaoContrato
covid
dtCompetencia
vlBase
vlProvento
vlDesconto
vlLiquido
custom
itens


In [6]:
mapa_colunas_servidores = {
    # Identificadores (High Cardinality)
    "numMatricula": "id_matricula_servidor",

    # Variáveis Categóricas de Texto (Categorical Features)
    "descServidor": "txt_nome_servidor",
    "descUnidade": "txt_unidade_lotacao",
    "descCargo": "txt_cargo_base",
    "descFuncao": "txt_funcao_confianca",
    "descVinculo": "txt_tipo_vinculo",
    "descTpFolha": "txt_tipo_folha",
    "descSituacaoContrato": "txt_situacao_contrato",

    # Flags Binárias (Binary Features)
    "covid": "bool_vinculo_covid",

    # Variáveis Temporais (Time Features)
    "dtCompetencia": "dt_competencia_folha",

    # Variáveis Contínuas Numéricas (Continuous Features)
    "vlBase": "val_salario_base",
    "vlProvento": "val_total_proventos",
    "vlDesconto": "val_total_descontos",
    "vlLiquido": "val_salario_liquido"  # CORRIGIDO: De 'desconto' para 'liquido'
}

df.rename(columns=mapa_colunas_servidores, inplace=True)

In [7]:
for column in df.columns:
  print(column)

id_matricula_servidor
txt_nome_servidor
txt_unidade_lotacao
txt_cargo_base
txt_funcao_confianca
txt_tipo_vinculo
txt_tipo_folha
txt_situacao_contrato
bool_vinculo_covid
dt_competencia_folha
val_salario_base
val_total_proventos
val_total_descontos
val_salario_liquido
custom
itens


In [8]:
df_por_cargo = df.groupby('txt_cargo_base').size().sort_values(ascending=False)
cargos = list(df_por_cargo.index)
smh = transmutar_lista_cargos(cargos)
print(smh)

[{'cargo_original': 'PROFESSOR MUNICIPAL II', 'txt_cargo_base': 'PROFESSOR MUNICIPAL', 'val_carga_horaria': 'NÃO ESPECIFICADO'}, {'cargo_original': 'MONITOR DE APOIO A EDUCACAO 40H', 'txt_cargo_base': 'MONITOR DE APOIO A EDUCACAO', 'val_carga_horaria': '40H'}, {'cargo_original': 'AUXILIAR DE SERVICO PUBLICO 30H', 'txt_cargo_base': 'AUXILIAR DE SERVICO PUBLICO', 'val_carga_horaria': '30H'}, {'cargo_original': 'MONITOR DE APOIO A EDUCACAO 35H', 'txt_cargo_base': 'MONITOR DE APOIO A EDUCACAO', 'val_carga_horaria': '35H'}, {'cargo_original': 'TECNICO EM ENFERMAGEM 12X36', 'txt_cargo_base': 'TECNICO EM ENFERMAGEM', 'val_carga_horaria': '12X36'}, {'cargo_original': 'AGENTE PUBLICO ADMINISTRATIVO 40H', 'txt_cargo_base': 'AGENTE PUBLICO ADMINISTRATIVO', 'val_carga_horaria': '40H'}, {'cargo_original': 'AUXILIAR DE SERVICO PUBLICO 30', 'txt_cargo_base': 'AUXILIAR DE SERVICO PUBLICO', 'val_carga_horaria': '30'}, {'cargo_original': 'AG COMUNITARIO SAUDE ACS (PS)', 'txt_cargo_base': 'AG COMUNITARIO

In [9]:
cargos_def = [item['txt_cargo_base'] for item in smh]
carga_horaria = [item['val_carga_horaria'] for item in smh]

if len(cargos_def) == len(cargos):
  df['val_carga_horaria'] = df["txt_cargo_base"].map(dict(zip(cargos, carga_horaria)))
  df.txt_cargo_base.replace(dict(zip(cargos, cargos_def)), inplace=True)

/tmp/ipykernel_3281/555399051.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df.txt_cargo_base.replace(dict(zip(cargos, cargos_def)), inplace=True)


In [10]:
df_por_vinculo = df.groupby('txt_tipo_vinculo').count().sort_values(by='id_matricula_servidor', ascending=False)
df_por_unid = df.groupby('txt_unidade_lotacao').count().sort_values(by='id_matricula_servidor', ascending=False)
df_por_cargo = df.groupby('txt_cargo_base').count().sort_values(by='id_matricula_servidor', ascending=False)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Criando a figura e os 3 eixos separadamente
fig, ax = plt.subplots(3, 1, figsize=(15,30), dpi=300)

# Gráfico 1: Vínculos
df_por_vinculo = df_por_vinculo[:5]
ax[0].bar(df_por_vinculo.index.values, df_por_vinculo['id_matricula_servidor'], color = 'blue')
ax[0].set_title('Vínculos')
ax[0].tick_params(axis='x', rotation=45)
for i, v in enumerate(df_por_vinculo['id_matricula_servidor']):
    ax[0].text(i, v, str(v), ha='center', va='bottom', fontdict=dict(color='blue', weight='bold', size=15))

# Gráfico 2: Unidades
ax[1].bar(df_por_unid.index.values, df_por_unid['id_matricula_servidor'], color='orange')
ax[1].set_title('Unidades')
ax[1].tick_params(axis='x', rotation=90)
for i, v in enumerate(df_por_unid['id_matricula_servidor']):
    ax[1].text(i, v, str(v), ha='center', va='bottom', fontdict=dict(color='orange', weight='bold', size=15))

# Gráfico 3: Cargos (Top 15 para não poluir o visual)
top_cargos = df_por_cargo.head(5)
ax[2].bar(top_cargos.index.values, top_cargos['id_matricula_servidor'], color='green')
ax[2].set_title('Top 5 Cargos')
ax[2].tick_params(axis='x', rotation=45)
for i, v in enumerate(top_cargos['id_matricula_servidor']):
    ax[2].text(i, v, str(v), ha='center', va='bottom', fontdict=dict(color='green', weight='bold', size=15))

plt.tight_layout()
plt.show()

In [ ]:
df.info

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['val_salario_base', 'val_total_proventos', 'val_total_descontos', 'val_salario_liquido']),
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['txt_cargo_base'])
    ])

ssc = StandardScaler()
df_transformed = ssc.fit_transform(df[['val_salario_base', 'val_total_proventos', 'val_total_descontos', 'val_salario_liquido']])
df_transformed = pd.DataFrame(df_transformed, columns=['val_salario_base', 'val_total_proventos', 'val_total_descontos', 'val_salario_liquido'])
df_transformed.head()
